# Lesson 02 - 探索 Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** 是一個用於構建 AI 代理的統一框架。它提供了一個乾淨且可組合的架構，包含四個核心組件：

- **Client** – 連接到 AI 模型端點並處理通訊
- **Agent** – 使用指令和工具定義包裝客戶端
- **Tools** – 透過模型可呼叫的自訂函數擴展代理能力
- **Session** – 維護對話歷史以支援多輪互動

在本課中，我們將使用這些概念建立一個**旅遊預訂代理**，用於檢查目的地的可用性。


## 設定


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## 了解代理框架架構

Microsoft 代理框架遵循分層架構：

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **客戶端** – `AzureAIProjectAgentProvider` 連接到 Azure OpenAI 部署。它負責身份驗證、請求格式化和回應解析。
2. **代理** – 由客戶端透過 `provider.create_agent()` 創建，該代理結合模型存取、指令（系統提示）和工具。
3. **工具** – 使用 `@tool` 裝飾的 Python 函數，代理可以調用這些函數來執行操作或檢索資料。
4. **會話** – `AgentSession` 物件（由 `agent.create_session()` 創建），存儲對話歷史，使得代理能記住先前上下文並進行多輪對話。

讓我們一步步建立每一層。


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 使用 @tool 裝飾器新增工具

工具讓代理人執行生成文字之外的動作。`@tool` 裝飾器將一般的 Python 函式轉換成代理人可以呼叫的功能。

重點：
- 使用 `Annotated[type, "description"]` 讓模型了解每個參數。
- 文件字串會成為模型看到的工具描述。
- `approval_mode="never_require"` 表示工具會自動執行，無需用戶確認。


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## 使用工具建立代理人

現在我們將客戶端、指令與工具結合成一個代理人。`instructions` 作為系統提示 — 它們定義了代理人的角色與行為。


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## 多輪對話與會話

`AgentSession`（透過 `agent.create_session()` 建立）會追蹤對話中的所有訊息。透過在每次呼叫 `agent.run()` 時傳遞相同的會話，代理可以存取完整的對話歷史並參考先前的訊息。

我們傳入 `tools=[check_destination_availability]`，讓代理在每一輪都能呼叫我們的可用性檢查工具。


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Summary

在本課程中，您探索了 Microsoft Agent Framework 的四大支柱：

| Concept | What You Learned |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` 使用憑證驗證連接到 Azure OpenAI |
| **Agent** | `provider.create_agent()` 將模型連接與指令和名稱捆綁在一起 |
| **Tools** | `@tool` 裝飾器用於暴露供代理調用的 Python 函數 |
| **Session** | `agent.create_session()` 維護多輪對話歷史 |

這些構建塊組合在一起，創建可以進行自然對話、調用外部函數並維持上下文的代理——為後續課程中的更高級代理模式奠定基礎。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件係使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 所翻譯。雖然我們力求準確，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件之母語版本應視為權威來源。對於重要資訊，建議採用專業人工翻譯。我們不對因使用本翻譯內容而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
